<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 4 — Intégration et Contrôles Inter-sources

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif

Construire le **dataset analytique final** `dataset_complet.csv` en fusionnant
les trois bases nettoyées des Phases 1, 2 et 3.

Ce dataset constitue la **base unique** pour toutes les analyses ultérieures (Phases 5 & 6).

**Stratégie de jointure :**
```
dossier_nettoye (98 935 lignes — unité d'analyse)
    ├── LEFT JOIN temps_agrégé       (sur Numero_dossier_ID = Numero.dossier)
    └── LEFT JOIN ressources         (sur Matricule.de.traitement = Matricule)
```

Le LEFT JOIN depuis `dossier` garantit que **toutes les 98 935 lignes** sont conservées.


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine (présence de .git ou requirements.txt)."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))
from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
# --- Chargement des trois bases nettoyées
dossier    = pd.read_csv('data/dossier_nettoye.csv',     encoding='utf-8')
ressources = pd.read_csv('data/ressources_nettoyees.csv', encoding='utf-8')
temps      = pd.read_csv('data/temps_nettoye.csv',        encoding='utf-8')

print(f" dossier    : {dossier.shape[0]:,} lignes × {dossier.shape[1]} colonnes")
print(f" ressources : {ressources.shape[0]:,} lignes × {ressources.shape[1]} colonnes")
print(f" temps      : {temps.shape[0]:,} lignes × {temps.shape[1]} colonnes")
print()
print("Clés de jointure :")
print(f"  dossier.Numero_dossier_ID    ↔ temps.Numero.dossier")
print(f"  dossier.Matricule.de.traitement ↔ ressources.Matricule")

---
## Section 1 — D1 : Agrégation de `temps` par dossier

### Objectif
Synthétiser toutes les interventions d'un dossier en une seule ligne d'indicateurs.

### Décision : LEFT JOIN — conservation des dossiers sans intervention
> Les 2 180 dossiers sans intervention dans `temps` obtiendront des NaN pour ces variables.  
> NaN ≠ durée nulle : l'absence d'enregistrement est une information distincte de zéro.

In [ ]:
# --- Agrégation de temps par dossier
temps_agrege = (
    temps.groupby('Numero.dossier')
    .agg(
        nb_interventions          = ('Matricule', 'count'),
        nb_agents_distincts       = ('Matricule', 'nunique'),
        duree_totale_min          = ('duree.corrigee', 'sum'),
        duree_moyenne_min         = ('duree.corrigee', 'mean'),
        date_premiere_intervention = ('Date.debut.traitement', 'min'),
        date_derniere_intervention = ('Date.debut.traitement', 'max'),
    )
    .reset_index()
    .rename(columns={'Numero.dossier': 'Numero_dossier_ID'})
)

print(f" Agrégat temps : {len(temps_agrege):,} dossiers uniques")
print()
print("Statistiques de l'agrégat :")
print(temps_agrege[['nb_interventions','nb_agents_distincts',
                     'duree_totale_min','duree_moyenne_min']].describe().round(2))
print()
print("Aperçu :")
temps_agrege.head(5).style_duplicates()

In [ ]:
# Vérification : nb_interventions vs nb_agents_distincts
dossiers_multi_agents = (temps_agrege['nb_agents_distincts'] > 1).sum()
dossiers_un_agent     = (temps_agrege['nb_agents_distincts'] == 1).sum()
print(f"Dossiers traités par 1 seul agent    : {dossiers_un_agent:,}")
print(f"Dossiers traités par plusieurs agents : {dossiers_multi_agents:,}")
print()
print("Distribution du nb d'agents par dossier :")
print(temps_agrege['nb_agents_distincts'].value_counts().sort_index().head(10))

---
## Section 2 — D2 : Extraction des caractéristiques de l'agent depuis `ressources`

### Stratégie
Pour chaque dossier, récupérer les caractéristiques de l'agent `Matricule.de.traitement`
à la date la plus proche de `date.ouverture`.

**Algorithme :**
1. Pour chaque dossier, récupérer son `Matricule.de.traitement` et sa `date.ouverture`
2. Chercher dans `ressources` la ligne pour ce matricule avec `Date.presence <= date.ouverture`
3. Prendre la ligne avec la `Date.presence` la plus récente (la plus proche de la date d'ouverture)
4. Récupérer les colonnes souhaitées

In [ ]:
# --- Préparation de la jointure par merge_asof
# merge_asof nécessite une clé numérique ou datetime — on convertit Date.presence
ressources_triees = ressources.copy()
ressources_triees['Date.presence'] = pd.to_datetime(ressources_triees['Date.presence'], format='%Y/%m/%d')
ressources_triees = ressources_triees.sort_values('Date.presence').reset_index(drop=True)

# Préparation du côté dossier
dossier_cles = dossier[['Numero_dossier_ID', 'Matricule.de.traitement', 'date.ouverture']].copy()
dossier_cles.rename(columns={'Matricule.de.traitement': 'Matricule',
                              'date.ouverture': 'Date.presence'}, inplace=True)
dossier_cles['Date.presence'] = pd.to_datetime(dossier_cles['Date.presence'], format='%Y/%m/%d')
dossier_cles = dossier_cles.sort_values('Date.presence').reset_index(drop=True)

# Colonnes à récupérer depuis ressources
colonnes_agent = ['Matricule', 'Date.presence', 'Lieu.travail', 'Population',
                  'Site', 'Type.de.contrat', 'Duree.travail', 'Temps.travail', 'Experience']

# merge_asof backward : pour chaque dossier, prendre la présence la plus récente ≤ date.ouverture
caracteristiques_agents = pd.merge_asof(
    dossier_cles,
    ressources_triees[colonnes_agent],
    on='Date.presence',
    by='Matricule',
    direction='backward'
)

# Renommage pour clarté dans le dataset final
caracteristiques_agents = caracteristiques_agents.rename(columns={
    'Lieu.travail'    : 'agent_lieu_travail',
    'Population'      : 'agent_population',
    'Site'            : 'agent_site',
    'Type.de.contrat' : 'agent_contrat',
    'Duree.travail'   : 'agent_duree_travail_j',
    'Temps.travail'   : 'agent_temps_travail_pct',
    'Experience'      : 'agent_experience_j',
})

print(f" Caractéristiques agents extraites : {len(caracteristiques_agents):,} lignes")
print()
n_nan_agent = caracteristiques_agents['agent_experience_j'].isna().sum()
print(f"Dossiers sans présence agent trouvée : {n_nan_agent:,}")
print()
print("Aperçu :")
caracteristiques_agents[['Numero_dossier_ID','Matricule','agent_lieu_travail',
                          'agent_contrat','agent_experience_j']].head(5).style_duplicates()

---
## Section 3 — Jointure principale (LEFT JOIN depuis `dossier`)

### Étapes
1. `dossier` ← `temps_agrege` (sur `Numero_dossier_ID`)
2. résultat ← `caracteristiques_agents` (sur `Numero_dossier_ID`)

In [ ]:
# --- Étape 3a : dossier ← temps_agrege
dataset = dossier.merge(
    temps_agrege,
    on='Numero_dossier_ID',
    how='left'
)

print(f"Après jointure dossier ← temps : {len(dataset):,} lignes")
assert len(dataset) == len(dossier), f" Divergence : {len(dataset)} ≠ {len(dossier)}"
print(" Vérification : unicité préservée ")

In [ ]:
# --- Étape 3b : dataset ← caracteristiques_agents
colonnes_agent_a_joindre = [
    'Numero_dossier_ID', 'agent_lieu_travail', 'agent_population',
    'agent_site', 'agent_contrat', 'agent_duree_travail_j',
    'agent_temps_travail_pct', 'agent_experience_j'
]

dataset = dataset.merge(
    caracteristiques_agents[colonnes_agent_a_joindre],
    on='Numero_dossier_ID',
    how='left'
)

print(f"Après jointure dataset ← ressources : {len(dataset):,} lignes × {dataset.shape[1]} colonnes")
assert len(dataset) == len(dossier), f" Divergence : {len(dataset)} ≠ {len(dossier)}"
print(" Vérification : unicité préservée ")

---
## Section 4 — D3 : Contrôles post-jointure

### Vérification des NaN introduits par les jointures

In [ ]:
# --- Bilan NaN post-jointure
nan_post_jointure = dataset.isna().sum()
nan_post_jointure = nan_post_jointure[nan_post_jointure > 0].sort_values(ascending=False)

df_nan_bilan = pd.DataFrame({
    'Colonne'       : nan_post_jointure.index,
    'NaN'           : nan_post_jointure.values,
    '% du total'    : (nan_post_jointure.values / len(dataset) * 100).round(2),
    'Origine'       : ['Phase 1-3' if c in dossier.columns else
                       'Jointure temps' if c in temps_agrege.columns else
                       'Jointure ressources'
                       for c in nan_post_jointure.index]
})
print(f"Valeurs manquantes dans dataset_complet ({len(dataset):,} lignes) :")
df_nan_bilan

In [ ]:
# --- Contrôles d'intégrité
print("=== CONTRÔLES D'INTÉGRITÉ ===")
print()

# 1. Unicité de la clé primaire
n_doublons_id = dataset['Numero_dossier_ID'].duplicated().sum()
print(f"1. Doublons Numero_dossier_ID : {n_doublons_id}")
assert n_doublons_id == 0, " La clé primaire n'est plus unique !"
print("    Clé primaire unique")

# 2. Dossiers sans intervention temps
n_sans_temps = dataset['nb_interventions'].isna().sum()
pct_sans_temps = n_sans_temps / len(dataset) * 100
print(f"2. Dossiers sans intervention temps : {n_sans_temps:,} ({pct_sans_temps:.1f}%)")

# 3. Dossiers sans données agent
n_sans_agent = dataset['agent_experience_j'].isna().sum()
print(f"3. Dossiers sans données agent (matricule orphelin) : {n_sans_agent:,}")

# 4. Plage temporelle
print(f"4. Plage date.ouverture : {dataset['date.ouverture'].min()} → {dataset['date.ouverture'].max()}")
print()
print(" Tous les contrôles passés")

---
## Section 5 — Variables dérivées

### Objectif
Calculer des variables supplémentaires utiles pour l'analyse économétrique et ML.